# D-LinkNet-ResNet50：v6 overlap40 正式基线

这是**真正的 D-LinkNet**（空洞卷积中心块），不是 SMP `Linknet`。固定5通道、seed=42、80 epochs，并按 `val_mIoU_fg` 保存最佳权重。物理 batch=2、梯度累积2次，有效 batch=4。

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata, json, shutil
from pathlib import Path

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/yuanssy/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/v6data/dataset_v6_random811_overlap40'),
]
print('Python:', sys.version)

## 1. 环境检查

In [ ]:
required = [('rasterio', 'rasterio'), ('matplotlib', 'matplotlib'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if importlib.util.find_spec('segmentation_models_pytorch') is None or smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import torch, rasterio, segmentation_models_pytorch as smp
assert torch.cuda.is_available(), '请在 Kaggle Notebook settings 中开启 GPU'
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('SMP:', smp.__version__, 'Rasterio:', rasterio.__version__)
x = torch.randn(512, 512, device='cuda')
print('CUDA test:', float((x @ x).mean()))
del x
torch.cuda.empty_cache()

## 2. 获取代码

In [ ]:
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录存在但不是Git仓库: {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
assert (PROJECT_DIR / 'models' / 'dlinknet.py').is_file(), '当前代码版本没有D-LinkNet，请先拉取最新提交'
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('Project:', PROJECT_DIR)
print('Commit:', commit)

## 3. 核验 overlap40 数据版本

In [ ]:
import numpy as np
EXPECTED_TILES = {'train': 1598, 'val': 200, 'test': 200}
EXPECTED_MEAN = np.array([0.15665339073973303, 0.6052870962271574, 0.22171011101838023, 0.5087022443378417, 0.46687463729626205])
EXPECTED_STD = np.array([0.07239327406001447, 0.35159567816693277, 0.23999408652260576, 0.18305312443820845, 0.18653673179588806])

def tif_files(folder):
    return sorted([*folder.glob('*.tif'), *folder.glob('*.tiff')])

existing = [p for p in DATA_CANDIDATES if p.is_dir()]
assert existing, '没有找到 overlap40 数据集，请检查 Add data'
DATA_ROOT = existing[0]
for split, expected in EXPECTED_TILES.items():
    images = tif_files(DATA_ROOT / split / 'image')
    masks = tif_files(DATA_ROOT / split / 'mask')
    assert len(images) == len(masks) == expected, (split, len(images), len(masks))
    assert {p.stem for p in images} == {p.stem for p in masks}, f'{split} image-mask不匹配'
    print(split, len(images))
stats = json.loads((DATA_ROOT / 'normalization_stats.json').read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], EXPECTED_MEAN, rtol=0, atol=1e-12)
assert np.allclose(stats['std'], EXPECTED_STD, rtol=0, atol=1e-12)
print('Data:', DATA_ROOT)
print('数据版本核验通过')

## 4. 模型和显存冒烟测试

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
from models.dlinknet import DLinkNet
smoke_model = DLinkNet('resnet50', None, in_channels=5, classes=5).cuda().train()
print('Parameters:', f'{sum(p.numel() for p in smoke_model.parameters()):,}')
try:
    smoke_x = torch.randn(2, 5, 512, 512, device='cuda')
    with torch.amp.autocast('cuda'):
        smoke_y = smoke_model(smoke_x)
        smoke_loss = smoke_y.mean()
    smoke_loss.backward()
    print('batch=2 smoke shape:', tuple(smoke_y.shape))
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError('batch=2显存不足：将下一格BATCH_SIZE改为1、ACCUM_STEPS改为4后重新运行本格时也把smoke_x的2改成1') from exc
finally:
    del smoke_model
    for name in ('smoke_x', 'smoke_y', 'smoke_loss'):
        if name in globals(): del globals()[name]
    torch.cuda.empty_cache()

## 5. 正式训练配置

In [ ]:
MODEL = 'DLinkNet'
ENCODER = 'resnet50'
SEED = 42
EPOCHS = 80
MAX_STEPS = 0
NUM_WORKERS = 2
BATCH_SIZE = 2
ACCUM_STEPS = 2
RUN_SUFFIX = 'formal80_valfg'
assert BATCH_SIZE * ACCUM_STEPS == 4
run_name = f'v6_overlap40_{MODEL}_seed{SEED}_{RUN_SUFFIX}'
result_dir = OUTPUT_ROOT / f'result_{run_name}'
assert not result_dir.exists(), f'结果目录已存在，为防覆盖请修改RUN_SUFFIX: {result_dir}'
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_baseline.py'),
    '--model', MODEL, '--encoder', ENCODER,
    '--data-dir', str(DATA_ROOT), '--output-dir', str(OUTPUT_ROOT),
    '--seed', str(SEED), '--epochs', str(EPOCHS), '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS), '--batch-size', str(BATCH_SIZE),
    '--accum-steps', str(ACCUM_STEPS), '--run-name', run_name,
    '--skip-test-evaluation',
]
print(' '.join(command))
print('Output:', result_dir)

## 6. 开始正式训练

In [ ]:
subprocess.check_call(command, cwd=PROJECT_DIR)

## 7. 统一评估并打包

In [ ]:
metrics_path = result_dir / 'metrics.json'
checkpoint_path = result_dir / 'best_model.pth'
assert metrics_path.is_file() and checkpoint_path.is_file()
result = json.loads(metrics_path.read_text(encoding='utf-8'))
assert result['selection_metric'] == 'val_mIoU_fg', result['selection_metric']
print(json.dumps(result, ensure_ascii=False, indent=2))
assert result['test'] is None and result['automatic_test_evaluation'] is False
evaluation_dir = result_dir / 'val_diagnostics'
eval_command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'evaluate_segmentation.py'),
    '--model', MODEL, '--encoder', ENCODER, '--data-dir', str(DATA_ROOT),
    '--checkpoint', str(checkpoint_path), '--output-dir', str(evaluation_dir),
    '--num-workers', str(NUM_WORKERS), '--samples-per-group', '4', '--split', 'val',
]
subprocess.check_call(eval_command, cwd=PROJECT_DIR)
archive_path = shutil.make_archive(str(OUTPUT_ROOT / result_dir.name), 'zip', root_dir=result_dir)
print('请下载:', archive_path)